# Lab 11: CNN Model for Time-Series Forecasting

**Student Name:** Malik Awais Bashir  
**Registration Number:** 22JZELE0474  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus  

## Goal of this Lab
* Import TensorFlow/Keras libraries, Scikit-learn evaluation metrics, and supporting time-series utility modules.
* Design a CNN model with Conv1D layers for processing sequential forecasting data.
* Set up the optimizer, callbacks, model checkpoints, and training configurations.
* Load the training, validation, and testing datasets for time-series prediction tasks.
* Measure the forecasting performance of the CNN model using commonly used regression evaluation metrics.


## Section 1: Working Directory and Library Imports
This section sets the Lab 10/11 path and imports all libraries required for CNN-based time-series forecasting.


In [2]:
import os
os.chdir(r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11')

In [3]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [4]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [5]:
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

In [6]:
model1 = CNN()
model1.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 24, 21)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 23, 16)         │           688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 22, 16)         │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 352)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           353 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,569 (6.13 KB)

 Trainable params: 1,569 (6.13 KB)

 Non-trainable params: 0 (0.00 B)

## Section 2: Model Parameters and CNN Architecture
The following cells define input shape parameters and build the Conv1D-based CNN model for forecasting.


In [7]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [8]:
checkpoints = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [9]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

In [10]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =CNN()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [11]:
import os
path_dataset =r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

c:\Users\hp\anaconda3\envs\dsp\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

## Section 3: Training Configuration and Callback Setup
This section prepares checkpoint saving, monitoring, optimizer configuration, and model compilation for training.


In [12]:
time_steps=24
num_features=21

In [13]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.009746551513671875 sec


In [14]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,verbose = verbose)

Epoch 1/10
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6429 - mae: 0.6429 - mape: 264.6394
Epoch 1: val_loss improved from None to 0.10509, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0001-loss0.11.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 6s 76ms/step - loss: 0.3525 - mae: 0.3525 - mape: 166.0435 - val_loss: 0.1051 - val_mae: 0.1051 - val_mape: 33.6077
Epoch 2/10
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1097 - mae: 0.1097 - mape: 40.0541
Epoch 2: val_loss improved from 0.10509 to 0.04128, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0002-loss0.04.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 0.0944 - mae: 0.0944 - mape: 38.5845 - val_loss: 0.0413 - val_mae: 0.0413 - val_mape: 16.2020
Epoch 3/10
18/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0714 - mae: 0.0714 - mape: 30.8094
Epoch 3: val_loss did not improve from 0.04128
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.0682 - mae: 0.0682 - mape: 31.8750 - val_loss: 0.0495 - val_mae: 0.0495 - val_mape: 14.9067
Epoch 4/10
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0580 - mae: 0.0580 - mape: 26.6716 
Epoch 4: val_loss did not improve from 0.04128
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 0.0591 - mae: 0.0591 - mape: 31.2313 - val_loss: 0.0535 - val_mae: 0.0535 - val_mape: 18.1350
Epoch 5/10
20/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0630 - mae: 0.0630 - mape: 28.9596
Epoch 5: val_loss did not improve from 0.04128
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - loss: 0.0540 - mae: 0.0540 - mape: 25.4033 - val_loss: 0.0537 - val_mae: 0.0537 - val_mape: 16.2542
Epoch

In [16]:

model = load_model(r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0002-loss0.04.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 395ms/step
Mean Absolute Error (MAE): 899.79
Median Absolute Error (MedAE): 545.49
Mean Squared Error (MSE): 1287218.97
Root Mean Squared Error (RMSE): 1134.56
Mean Absolute Percentage Error (MAPE): 5.78 %
Median Absolute Percentage Error (MDAPE): 3.52 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


In [20]:
checkpoints = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0002-loss0.04.h5'
model=r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0002-loss0.04.h5'
start_epoch= 58

## Section 4: Dataset Loading, Prediction, and Evaluation
The remaining cells load the data, prepare it for the CNN model, generate predictions, and calculate forecasting metrics.


In [21]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
model_path = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0002-loss0.04.h5'
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model_path))
    model = load_model(model_path, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0002-loss0.04.h5...
[INFO] old learning rate: 9.999999747378752e-05
[INFO] new learning rate: 9.999999747378752e-05


In [22]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/10


24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0730 - mae: 0.0730 - mape: 34.1296
Epoch 1: val_loss improved from None to 0.04368, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0002-loss0.04.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - loss: 0.0718 - mae: 0.0718 - mape: 31.4313 - val_loss: 0.0437 - val_mae: 0.0437 - val_mape: 16.8044
Epoch 2/10
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0676 - mae: 0.0676 - mape: 29.6248
Epoch 2: val_loss did not improve from 0.04368
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 0.0678 - mae: 0.0678 - mape: 30.8780 - val_loss: 0.0449 - val_mae: 0.0449 - val_mape: 16.3936
Epoch 3/10
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0620 - mae: 0.0620 - mape: 29.1143
Epoch 3: val_loss improved from 0.04368 to 0.04297, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0002-loss0.04.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - loss: 0.0645 - mae: 0.0645 - mape: 29.8016 - val_loss: 0.0430 - val_mae: 0.0430 - val_mape: 14.6504
Epoch 4/10
21/27 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0642 - mae: 0.0642 - mape: 31.4103
Epoch 4: val_loss did not improve from 0.04297
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 0.0614 - mae: 0.0614 - mape: 28.3708 - val_loss: 0.0439 - val_mae: 0.0439 - val_mape: 14.5129
Epoch 5/10
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0613 - mae: 0.0613 - mape: 28.2101
Epoch 5: val_loss did not improve from 0.04297
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - loss: 0.0601 - mae: 0.0601 - mape: 28.6167 - val_loss: 0.0502 - val_mae: 0.0502 - val_mape: 16.1802
Epoch 6/10
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0584 - mae: 0.0584 - mape: 30.2571
Epoch 6: val_loss did not improve from 0.04297
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - loss: 0.0578 - mae: 0.0578 - mape: 28.3951 - val_loss: 0.0471 - val_mae: 0.0471 - val_mape: 14.8142
Epoch 

In [24]:

model = load_model(r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 10,11\E1-cp-0002-loss0.04.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 423ms/step
Mean Absolute Error (MAE): 1943.91
Median Absolute Error (MedAE): 1750.93
Mean Squared Error (MSE): 4536799.63
Root Mean Squared Error (RMSE): 2129.98
Mean Absolute Percentage Error (MAPE): 12.42 %
Median Absolute Percentage Error (MDAPE): 10.83 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


## Conclusion
* In this lab, a CNN model was implemented for time-series forecasting applications.
* Conv1D layers were utilized to capture local temporal features from sequential data.
* The model was trained to learn patterns and trends present in the time-series dataset.
* Forecasting results were generated and compared with actual values.
* The model's predictive performance was assessed using standard regression evaluation metrics.
